<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/English_to_Ukrainian_Colabwhisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Fast Translator with AI Voice Feedback (No API Key)
# @markdown This version uses Google's translation engine and Gemini TTS for voice status updates.

import ipywidgets as widgets
from IPython.display import display, Javascript, Audio
import io
import os
import requests
import json
import base64
import time

# --- INSTALL DEPENDENCIES ---
try:
    from docx import Document
except ImportError:
    os.system('pip install python-docx -q')
    from docx import Document

def speak_status(text):
    """
    Triggers a browser-native Speech Synthesis (TTS)
    to give audio feedback on what the script is doing.
    """
    js_code = f"""
    var msg = new SpeechSynthesisUtterance({json.dumps(text)});
    window.speechSynthesis.speak(msg);
    """
    display(Javascript(js_code))

def google_translate(text, src='en', tgt='uk'):
    try:
        url = "https://translate.googleapis.com/translate_a/single"
        params = {
            "client": "gtx",
            "sl": src,
            "tl": tgt,
            "dt": "t",
            "q": text
        }
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            result_parts = response.json()[0]
            translated_text = "".join([part[0] for part in result_parts if part[0]])
            return translated_text
        return None
    except Exception:
        return None

def translate_text(btn):
    text = ""
    file_suffix = "en_to_ua"

    output_text.value = ""
    output_label.value = "⏳ Initializing..."
    translate_btn.disabled = True
    translate_btn.description = "Processing..."

    # Voice Feedback
    speak_status("Starting translation. Please wait.")

    # 1. Handle File Upload
    if file_upload.value:
        try:
            file_info = list(file_upload.value.values())[0]
            file_name = file_info['metadata']['name']
            content = file_info['content']

            if file_name.endswith('.docx'):
                doc = Document(io.BytesIO(content))
                text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
            else:
                text = content.decode('utf-8').strip()
        except Exception as e:
            output_label.value = f"❌ File Error: {str(e)}"
            speak_status("There was an error reading your file.")
            translate_btn.disabled = False
            return
    else:
        text = input_text.value.strip()

    if not text:
        output_label.value = "⚠️ Please upload a file or enter text."
        speak_status("Please provide some text to translate.")
        translate_btn.disabled = False
        return

    # 2. Process Translation
    try:
        direction = direction_dropdown.value
        src_code = 'en' if direction == "English to Ukrainian" else 'uk'
        tgt_code = 'uk' if direction == "English to Ukrainian" else 'en'
        file_suffix = "en_to_ua" if direction == "English to Ukrainian" else "ua_to_en"

        output_label.value = "⚙️ Translating via Google..."

        paragraphs = text.split('\n')
        translated_paragraphs = []

        for p in paragraphs:
            if p.strip():
                result = google_translate(p, src_code, tgt_code)
                translated_paragraphs.append(result if result else "[Failed]")
            else:
                translated_paragraphs.append("")

        final_result = "\n".join(translated_paragraphs)
        output_text.value = final_result

        # 3. Create .docx
        out_doc = Document()
        for line in translated_paragraphs:
            if line.strip():
                out_doc.add_paragraph(line)

        doc_io = io.BytesIO()
        out_doc.save(doc_io)
        doc_io.seek(0)

        # 4. Trigger Download
        b64 = base64.b64encode(doc_io.read()).decode()
        filename = f"translated_{file_suffix}.docx"
        js_download = f"""
        var element = document.createElement('a');
        element.setAttribute('href', 'data:application/vnd.openxmlformats-officedocument.wordprocessingml.document;base64,' + '{b64}');
        element.setAttribute('download', '{filename}');
        element.style.display = 'none';
        document.body.appendChild(element);
        element.click();
        document.body.removeChild(element);
        """
        display(Javascript(js_download))
        output_label.value = "✅ Success! Document downloaded."
        speak_status("Translation successful. Your document is ready for download.")

    except Exception as e:
        output_label.value = f"❌ Error: {str(e)}"
        speak_status("An error occurred during translation.")

    translate_btn.disabled = False
    translate_btn.description = "Start Fast Translation"

# --- UI SETUP ---
direction_dropdown = widgets.Dropdown(
    options=['English to Ukrainian', 'Ukrainian to English'],
    value='English to Ukrainian',
    description='Direction:',
    layout=widgets.Layout(width='300px')
)

file_upload = widgets.FileUpload(
    accept='.docx,.txt',
    multiple=False,
    description='Upload Document',
    button_style='info'
)

input_text = widgets.Textarea(
    placeholder='Paste text here...',
    layout=widgets.Layout(width='95%', height='180px')
)

output_text = widgets.Textarea(
    layout=widgets.Layout(width='95%', height='180px'),
    disabled=True
)

translate_btn = widgets.Button(
    description='Start Fast Translation',
    button_style='success',
    layout=widgets.Layout(width='300px', height='45px')
)

output_label = widgets.Label(value="Status: Ready")
translate_btn.on_click(translate_text)

ui = widgets.VBox([
    widgets.HTML("<h2 style='color: #1a73e8;'>Voice-Enabled Doc Translator</h2>"),
    widgets.HBox([direction_dropdown, file_upload]),
    input_text,
    translate_btn,
    output_label,
    output_text
])

display(ui)

<IPython.core.display.Javascript object>